# Task 3.1 — Ablation 1: Cross-Term Removal from $K_{DL}$

## What I am Ablating and Why

The paper's main contribution is the balanced kernel $K_{DL}$ which adds two cross-terms $k(a,d)$ and $k(b,c)$ to the plain kernel $K_D$. These cross-terms are what enforce symmetry. But do both contribute equally? And does half-symmetry give half the benefit?

**Component being ablated:** the cross-terms of $K_{DL}$ (Equation 6 of the paper).

**Three configurations I am testing:**
- Full $K_{DL}$: uses both cross-terms $k(a,d)$ and $k(b,c)$ — this is the full paper method
- Half $K_{DL}$: only one cross-term $k(a,d)$ retained — partial symmetry
- Plain $K_D$: no cross-terms at all — the baseline the paper compares against

**Hypothesis:** I expect a clear monotonic drop as I remove each cross-term. Both should contribute because Theorem 2 requires *both* for full symmetry. Removing one should be worse than the full version; removing both should be the worst.


## Ablation Component: The Cross-Terms in K_DL (Symmetry-Enforcing Terms)

### Component Description

The **Direct Linearised kernel** is defined as (**Section 3, Equation 6**):

$$K_{DL}((a,b),(c,d)) = \underbrace{k(a,c) + k(b,d)}_{\text{same as } K_D} + \underbrace{k(a,d) + k(b,c)}_{\text{cross-terms (our ablation target)}}$$

The **cross-terms** $k(a,d) + k(b,c)$ are the modification that the paper introduces to make the Direct Kernel symmetric. Without these two terms, $K_{DL}$ reduces to the plain $K_D$.

### Ablation Plan

I will compare three kernel variants:
1. **$K_{DL}$ (full)** — both direct terms + both cross-terms (paper's method)
2. **$K_{DL}$ with one cross-term removed** — $k(a,c) + k(b,d) + k(a,d)$ (partial symmetry)
3. **$K_D$ (no cross-terms)** — just $k(a,c) + k(b,d)$ (plain baseline)

This progressive removal lets us see the **marginal contribution** of each cross-term.

### Hypothesis

**I hypothesize that:**
- Removing **both cross-terms** will hurt accuracy the most, because the model loses the symmetry constraint entirely and may make inconsistent predictions.
- Removing **only one cross-term** will show a partial degradation — better than no cross-terms but worse than the full $K_{DL}$ — confirming that both cross-terms are necessary for proper symmetry.

This would validate the paper's design choice of using both $k(a,d)$ and $k(b,c)$, and not just one of them.

### Paper Citation

**Section 3, before and after Equation (6):** The paper notes that the linearised kernel is derived to ensure $K((a,b),(c,d)) = K((b,a),(c,d))$ for all inputs. It is the addition of cross-terms that achieves this.

**Theorem 2 (Section 3):** Formally proves that $K_{DL}$ is symmetric. The proof relies on the presence of **both** cross-terms $k(a,d)$ and $k(b,c)$. If only one is included, the symmetry proof fails.

In [1]:
import numpy as np
from sklearn.metrics.pairwise import rbf_kernel

# Verify manually that removing one cross-term breaks symmetry
np.random.seed(0)
a, b, c, d = [np.random.randn(5) for _ in range(4)]
G = 0.01

def K_DL_full(a, b, c, d):
    """Full balanced kernel — Eq. (6)"""
    return (rbf_kernel([a],[c],G) + rbf_kernel([b],[d],G)
          + rbf_kernel([a],[d],G) + rbf_kernel([b],[c],G)).item()

def K_DL_half(a, b, c, d):
    """Only one cross-term — breaks symmetry"""
    return (rbf_kernel([a],[c],G) + rbf_kernel([b],[d],G)
          + rbf_kernel([a],[d],G)).item()   # missing k(b,c)

def K_D(a, b, c, d):
    """No cross-terms — plain K_D"""
    return (rbf_kernel([a],[c],G) + rbf_kernel([b],[d],G)).item()

for name, fn in [('K_DL_full', K_DL_full), ('K_DL_half', K_DL_half), ('K_D', K_D)]:
    v_ab = fn(a, b, c, d)
    v_ba = fn(b, a, c, d)   # swapped input order
    sym  = np.isclose(v_ab, v_ba)
    print(f"{name:12s}: f(a,b)={v_ab:.4f}  f(b,a)={v_ba:.4f}  Symmetric={sym}")

K_DL_full   : f(a,b)=3.6954  f(b,a)=3.6954  Symmetric=True
K_DL_half   : f(a,b)=2.7190  f(b,a)=2.7933  Symmetric=False
K_D         : f(a,b)=1.8649  f(b,a)=1.8304  Symmetric=False


This confirms our ablation design is sound: only the **full $K_{DL}$ with both cross-terms** is symmetric, which is exactly what **Theorem 2** proves. Task 3.2 will test whether this matters for classification performance.